# OData Measurement and Test Result Queries (V2)

This notebook is the V2 companion of `../../query/query_measurements.ipynb`. It shows how to use OData queries to find test results, steps, measurements, and conditions in NI Measurement Data Services (V2 mode).

## Prerequisites

**⚠️ Important:** Run the `publish_sample_data.ipynb` notebook first to create the sample test data that this notebook queries.

- NI Measurement Data Services running in V2 mode
- Python environment with the `ni.datastore` package (see `../README.md`)
- Sample test data created by running `publish_sample_data.ipynb`

## Setup

Import the required libraries and create both metadata and data store clients.

In [ ]:
# Perform example-specific setup. This is not needed when writing production code.
# Set MDS_V2_GRPC_TARGET (for example, "[::1]:59606") to connect to a specific V2 service.
# When it is not set, the clients use discovery to find the service.
import os

import grpc

grpc_target = os.environ.get("MDS_V2_GRPC_TARGET", "")
# Remove gRPC's 4 MB receive limit so that large values, such as long waveforms, can be read.
grpc_options = [("grpc.max_receive_message_length", -1)]
grpc_channel = grpc.insecure_channel(grpc_target, options=grpc_options) if grpc_target else None

from ni.datastore.data.v2 import DataStoreClient
from ni.datastore.metadata import MetadataStoreClient

metadata_store_client = MetadataStoreClient(grpc_channel=grpc_channel)
data_store_client = DataStoreClient(grpc_channel=grpc_channel)

## Query Test Results

V2 can query test results directly. Test results also have a `status` that you can filter on.

**Additional test result query examples:**

```python
# Filter by status
done_test_results = data_store_client.query_test_results("$filter=Status eq 'Done'")
```

In [ ]:
print("=== Power Supply Test Results ===")
power_test_results = data_store_client.query_test_results("$filter=contains(Name,'Power Supply')")
for test_result in power_test_results:
    print(f"  📋 {test_result.name} - {test_result.status.name}, {test_result.outcome.name}")
assert len(power_test_results) >= 2

print("\n=== Failed Test Results ===")
failed_test_results = data_store_client.query_test_results("$filter=Outcome eq 'Failed'")
for test_result in failed_test_results:
    print(f"  ❌ {test_result.name}")
if not failed_test_results:
    print("  ✅ No failed test results found!")

## Query Measurements

Measurements are the individual data points collected during test steps. In V2, you can filter on the outcome in the query itself.

**Additional measurement query examples:**

```python
# Filter by measurement name
voltage_measurements = data_store_client.query_measurements("$filter=contains(Name,'Voltage')")
```

In [ ]:
from ni.datastore.data.v2 import Outcome

def print_measurement_with_outcome(measurement) -> str:
    name = measurement.name or "Unnamed Measurement"
    passed = measurement.outcome == Outcome.PASSED
    failed = measurement.outcome == Outcome.FAILED
    return f"{name} - {'✅ (Passed)' if passed else '❌ (Failed)' if failed else '❓ (Unknown)'}"

print("=== Failed Measurements ===")
failed_measurements = data_store_client.query_measurements("$filter=Outcome eq 'Failed'")
for measurement in failed_measurements[:15]:
    print(f"  ❌ {measurement.name} = {measurement.value}")
if not failed_measurements:
    print("  ✅ No failed measurements found!")
if len(failed_measurements) > 15:
    print(f"  ... and {len(failed_measurements) - 15} more measurements")
assert all(measurement.outcome == Outcome.FAILED for measurement in failed_measurements)

print("\n=== Voltage Measurements ===")
voltage_measurements = data_store_client.query_measurements("$filter=contains(Name,'Voltage')")
for measurement in voltage_measurements:
    print(f"  ⚡ {print_measurement_with_outcome(measurement)}")

print("\n=== Current Measurements ===")
current_measurements = data_store_client.query_measurements("$filter=contains(Name,'Current')")
for measurement in current_measurements:
    print(f"  🔌 {print_measurement_with_outcome(measurement)}")

print("\n=== Failed Measurements by Operator Alex Smith ===")
test_results_by_id = {}
for measurement in failed_measurements:
    if measurement.test_result_id not in test_results_by_id:
        test_results_by_id[measurement.test_result_id] = data_store_client.get_test_result(
            measurement.test_result_id
        )
    test_result = test_results_by_id[measurement.test_result_id]
    if test_result.operator_id:
        operator = metadata_store_client.get_operator(test_result.operator_id)
        if operator.name == "Alex Smith":
            print(f"  📋 {print_measurement_with_outcome(measurement)} - Operator: {operator.name}")

## Query Steps

Steps are the individual test phases within a test result. Each step can contain multiple measurements.

**Additional step query examples:**

```python
# Filter by step name
initialization_steps = data_store_client.query_steps("$filter=contains(Name,'Initialize')")
measurement_steps = data_store_client.query_steps("$filter=contains(Name,'Measure')")
```

In [ ]:
all_steps = data_store_client.query_steps("")

print(f"Found {len(all_steps)} steps total")
print("\nStep summary:")
for step in all_steps[:10]:  # Show first 10 steps
    print(f"  🔧 {step.name}")

if len(all_steps) > 10:
    print(f"  ... and {len(all_steps) - 10} more steps")

print("\n=== Steps containing 'Voltage' ===")
voltage_steps = data_store_client.query_steps("$filter=contains(Name,'Voltage')")
for step in voltage_steps:
    print(f"  🔧 {step.name}")

## Query Conditions

In V2, conditions belong to steps. To find conditions, query the steps that have them with `Conditions/any(...)`, then read each step's `conditions`.

`read_fields` limits which fields the service returns. When you use it, list every field that you need, such as `name` and `conditions`.

**Additional condition query examples:**

```python
# Find the measurements whose step has a humidity condition
humidity_measurements = data_store_client.query_measurements(
    "$filter=Conditions/any(c: c/Name eq 'Relative Humidity')"
)
```

In [ ]:
temp_and_pressure_steps = data_store_client.query_steps(
    "$filter=Conditions/any(c: contains(c/Name,'Temperature') or contains(c/Name,'Pressure'))",
    read_fields=["name", "conditions"],
)

unique_condition_names = sorted(
    {
        condition.name
        for step in temp_and_pressure_steps
        for condition in step.conditions
        if "Temperature" in condition.name or "Pressure" in condition.name
    }
)

if unique_condition_names:
    print(f"Found {len(temp_and_pressure_steps)} steps with temperature or pressure conditions")
    print("\nConditions related to temp and pressure:")
    for condition_name in unique_condition_names:
        print(f"  🌡️ {condition_name}")
else:
    print("No conditions found in the sample data")

print("\n=== Measurements whose step has a humidity condition ===")
humidity_measurements = data_store_client.query_measurements(
    "$filter=Conditions/any(c: c/Name eq 'Relative Humidity')"
)
print(f"  💧 {len(humidity_measurements)} measurements")

## Clean up

In [ ]:
metadata_store_client.close()
data_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
if grpc_channel is not None:
    grpc_channel.close()